# 29 — Gated B4/B5 ICH + Ordinal MLS Fusion

## Goal

Rapid DEV-only fusion of the strongest completed components:

- Notebook 18: EfficientNet-B4 2.5D ICH segmentation
- Notebook 22: EfficientNet-B5 2.5D ICH segmentation
- Notebook 26: dedicated 2.5D ICH presence gate
- Notebook 24: ordinal MLS model
- current fracture prediction from final-evaluation

No training is performed.

The search is staged:
1. continuity + B4/B5 volume fusion,
2. presence-gate threshold with a volume override to protect true hemorrhages,
3. ordinal MLS >=3 / >=5 calibration,
4. compare current fracture versus suppressing fracture entirely.

The locked TEST split is not used for tuning.

## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 2. Auto-discover saved outputs

In [ ]:
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
OUTPUT_ROOT = Path("/kaggle/working/gated_b4_b5_ich_ordinal_mls_fusion")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ICH_CLASSES = ["EDH", "SDH", "IPH", "SAH", "IVH"]
ICH_COLUMNS = [f"V_{x}" for x in ICH_CLASSES]

def all_files(name):
    out = []
    for root in SEARCH_ROOTS:
        if root.exists():
            out.extend(root.rglob(name))
    return list(dict.fromkeys(out))

def direct_answer_roots():
    roots = {}
    for path in all_files("00_DIRECT_ANSWERS.csv"):
        try:
            table = pd.read_csv(path)
        except Exception:
            continue
        text = " ".join(table.astype(str).fillna("").values.ravel()).lower()
        if "2.5d blood previous-center-next" in text and "expanded" in text:
            roots["b4"] = path.parent
        if "efficientnet-b5" in text and "2.5d" in text and "expanded" in text:
            roots["b5"] = path.parent
        if "efficientnet_b4" in text and "top5_mean" in text and "dev_recall" in " ".join(table.columns).lower():
            roots["gate"] = path.parent
        if "dev_bin_accuracy" in " ".join(table.columns).lower():
            roots["mls"] = path.parent
    return roots

roots = direct_answer_roots()
print("Discovered:", roots)

required = ["b4", "b5", "gate", "mls"]
missing = [x for x in required if x not in roots]
if missing:
    raise FileNotFoundError(f"Attach saved outputs for: {missing}")

def one_below(root, name):
    matches = list(root.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {name} below {root}, found {len(matches)}")
    return matches[0]

B4_SLICES = one_below(roots["b4"], "dev_slice_predictions.csv")
B5_SLICES = one_below(roots["b5"], "dev_slice_predictions.csv")
GATE_SCORES = one_below(roots["gate"], "dev_series_scores.csv")
MLS_SCORES = one_below(roots["mls"], "dev_series_scores.csv")

base_matches = all_files("common_dev_predictions.csv")
preferred = [p for p in base_matches if "final_evaluation" in str(p)]
if len(preferred) == 1:
    BASE_DEV = preferred[0]
elif len(base_matches) == 1:
    BASE_DEV = base_matches[0]
else:
    print("Base DEV candidates:", base_matches)
    raise FileNotFoundError("Could not uniquely identify common_dev_predictions.csv")

print("B4:", B4_SLICES)
print("B5:", B5_SLICES)
print("Gate:", GATE_SCORES)
print("MLS:", MLS_SCORES)
print("Base:", BASE_DEV)

## 3. Load tables

In [ ]:
def norm_id(v):
    try: return str(int(float(v)))
    except Exception: return str(v).strip()

b4 = pd.read_csv(B4_SLICES)
b5 = pd.read_csv(B5_SLICES)
gate = pd.read_csv(GATE_SCORES)
mls = pd.read_csv(MLS_SCORES)
base = pd.read_csv(BASE_DEV)

for df in [b4, b5, gate, mls, base]:
    df["series_id"] = df["series_id"].map(norm_id)

ids = set(base.series_id) & set(b4.series_id) & set(b5.series_id) & set(gate.series_id) & set(mls.series_id)
if len(ids) != 54:
    raise RuntimeError(f"Expected 54 DEV series, found {len(ids)}")

base = base[base.series_id.isin(ids)].sort_values("series_id").reset_index(drop=True)
gate = gate[gate.series_id.isin(ids)].sort_values("series_id").reset_index(drop=True)
mls = mls[mls.series_id.isin(ids)].sort_values("series_id").reset_index(drop=True)
print("DEV series:", len(ids))

## 4. Official triage rule

In [ ]:
def triage_one(vals):
    V_EDH=max(0.,float(vals["V_EDH"])); V_SDH=max(0.,float(vals["V_SDH"]))
    V_IPH=max(0.,float(vals["V_IPH"])); V_SAH=max(0.,float(vals["V_SAH"])); V_IVH=max(0.,float(vals["V_IVH"]))
    fracture=float(vals["fracture_prob"]); MLS=max(0.,float(vals["MLS_mm"]))
    total=V_EDH+V_SDH+V_IPH+V_SAH+V_IVH
    has_ich=total>=0.1; frac=fracture>=0.5

    if MLS>=5 and (has_ich or frac): return 2
    if V_EDH>=30: return 2
    if V_SDH>=70: return 2
    if V_IPH>=70: return 2
    if total>=60: return 2
    if has_ich and MLS>=3 and total>=40: return 2
    if frac and total>=15: return 2
    if MLS>=5 and not (has_ich or frac): return 1
    if has_ich: return 1
    if 3<=MLS<5: return 1
    if frac and total<15: return 1
    if total>=0.1 and MLS>=1: return 1
    return 0

def evaluate(ich, mls_values, frac_values):
    work = base[["series_id","true_triage"]].merge(ich,on="series_id").sort_values("series_id").reset_index(drop=True)
    pred=[]
    for i,row in enumerate(work.itertuples(index=False)):
        vals={c:float(getattr(row,c)) for c in ICH_COLUMNS}
        vals["MLS_mm"]=float(mls_values[i]); vals["fracture_prob"]=float(frac_values[i])
        pred.append(triage_one(vals))
    pred=np.asarray(pred)
    return float(f1_score(work.true_triage,pred,average="macro",labels=[0,1,2],zero_division=0)), float(accuracy_score(work.true_triage,pred)), pred

## 5. Continuity aggregation

In [ ]:
PROFILES = {
    "raw": {"EDH":1,"SDH":1,"IPH":1,"SAH":1,"IVH":1},
    "all2": {"EDH":2,"SDH":2,"IPH":2,"SAH":2,"IVH":2},
    "all3": {"EDH":3,"SDH":3,"IPH":3,"SAH":3,"IVH":3},
    "sah1_others2": {"EDH":2,"SDH":2,"IPH":2,"SAH":1,"IVH":2},
}

def subtype_volume(group, subtype, min_run):
    g=group.sort_values("slice_order")
    flags=g[f"pred_positive_{subtype}"].astype(bool).to_numpy()
    vols=g[f"pred_volume_{subtype}"].to_numpy(float)
    keep=np.zeros(len(g),bool); start=None
    for i in range(len(flags)+1):
        active=i<len(flags) and flags[i]
        if active and start is None: start=i
        if not active and start is not None:
            if i-start>=min_run: keep[start:i]=True
            start=None
    return float(vols[keep].sum())

def aggregate(df, profile):
    rows=[]
    for sid,g in df.groupby("series_id",sort=False):
        row={"series_id":sid}
        for s in ICH_CLASSES:
            row[f"V_{s}"]=subtype_volume(g,s,profile[s])
        rows.append(row)
    return pd.DataFrame(rows).sort_values("series_id").reset_index(drop=True)

agg_b4={name:aggregate(b4,p) for name,p in PROFILES.items()}
agg_b5={name:aggregate(b5,p) for name,p in PROFILES.items()}

## 6. Stage A — B4/B5 fusion

In [ ]:
current_mls=base["pred_MLS_mm"].to_numpy(float)
current_frac=base["pred_fracture_prob"].to_numpy(float)
ALPHAS=np.arange(0,1.01,0.1)

rows=[]
for p4,d4 in agg_b4.items():
    for p5,d5 in agg_b5.items():
        merged=d4.merge(d5,on="series_id",suffixes=("_b4","_b5"))
        for a in ALPHAS:
            fused=pd.DataFrame({"series_id":merged.series_id})
            for c in ICH_COLUMNS:
                fused[c]=a*merged[f"{c}_b4"]+(1-a)*merged[f"{c}_b5"]
            f1,acc,_=evaluate(fused,current_mls,current_frac)
            rows.append({"profile_b4":p4,"profile_b5":p5,"alpha_b4":float(a),"alpha_b5":float(1-a),"macro_F1":f1,"accuracy":acc})

fusion=pd.DataFrame(rows).sort_values(["macro_F1","accuracy"],ascending=False).reset_index(drop=True)
display(fusion.head(20))
best=fusion.iloc[0]

d4=agg_b4[best.profile_b4]; d5=agg_b5[best.profile_b5]
m=d4.merge(d5,on="series_id",suffixes=("_b4","_b5"))
best_ich=pd.DataFrame({"series_id":m.series_id})
for c in ICH_COLUMNS:
    best_ich[c]=float(best.alpha_b4)*m[f"{c}_b4"]+float(best.alpha_b5)*m[f"{c}_b5"]

## 7. Stage B — presence gate with volume override

In [ ]:
GATE_AGGS=["max","top2_mean","top3_mean","top5_mean","top10_mean"]
GATE_THRESHOLDS=np.arange(0.05,0.96,0.02)
OVERRIDE_TOTALS=[0.5,1.0,2.0,3.0,5.0,10.0,15.0,30.0,9999.0]

def apply_gate(ich, scores, threshold, override_total):
    out=ich.copy()
    totals=out[ICH_COLUMNS].sum(axis=1).to_numpy(float)
    negative=(scores<threshold) & (totals<override_total)
    out.loc[negative,ICH_COLUMNS]=0.0
    return out

gate_rows=[]
for agg in GATE_AGGS:
    scores=gate[agg].to_numpy(float)
    for th in GATE_THRESHOLDS:
        for override in OVERRIDE_TOTALS:
            gated=apply_gate(best_ich,scores,th,override)
            f1,acc,_=evaluate(gated,current_mls,current_frac)
            gate_rows.append({"aggregator":agg,"threshold":float(th),"override_total_mL":float(override),"macro_F1":f1,"accuracy":acc})

gate_search=pd.DataFrame(gate_rows).sort_values(["macro_F1","accuracy"],ascending=False).reset_index(drop=True)
gate_search.to_csv(OUTPUT_ROOT/"01_gate_search.csv",index=False)
display(gate_search.head(25))

bg=gate_search.iloc[0]
gated_ich=apply_gate(best_ich,gate[bg.aggregator].to_numpy(float),float(bg.threshold),float(bg.override_total_mL))

## 8. Stage C — ordinal MLS >=3 and >=5

In [ ]:
MLS_AGGS=["max","top3","top5","top10"]
TH=np.arange(0.05,0.96,0.02)
mls_rows=[]

for a3 in MLS_AGGS:
    p3=mls[f"p3_{a3}"].to_numpy(float)
    for t3 in TH:
        f3=p3>=t3
        for a5 in MLS_AGGS:
            p5=mls[f"p5_{a5}"].to_numpy(float)
            for t5 in TH:
                f5=p5>=t5
                values=np.zeros(len(mls),float); values[f3]=3.5; values[f5]=5.5
                f1,acc,_=evaluate(gated_ich,values,current_frac)
                mls_rows.append({"agg3":a3,"th3":float(t3),"agg5":a5,"th5":float(t5),"macro_F1":f1,"accuracy":acc})

mls_search=pd.DataFrame(mls_rows).sort_values(["macro_F1","accuracy"],ascending=False).reset_index(drop=True)
mls_search.to_csv(OUTPUT_ROOT/"02_mls_search.csv",index=False)
display(mls_search.head(20))

bm=mls_search.iloc[0]
f3=mls[f"p3_{bm.agg3}"].to_numpy(float)>=float(bm.th3)
f5=mls[f"p5_{bm.agg5}"].to_numpy(float)>=float(bm.th5)
best_mls=np.zeros(len(mls),float); best_mls[f3]=3.5; best_mls[f5]=5.5

## 9. Stage D — fracture ablation/calibration

In [ ]:
FRACTURE_OPTIONS=[]

FRACTURE_OPTIONS.append(("current_raw",current_frac))
FRACTURE_OPTIONS.append(("off",np.zeros(len(base),float)))

for th in np.arange(0.3,0.81,0.02):
    FRACTURE_OPTIONS.append((f"current_binary_{th:.2f}",(current_frac>=th).astype(float)))

frac_rows=[]
for name,values in FRACTURE_OPTIONS:
    f1,acc,_=evaluate(gated_ich,best_mls,values)
    frac_rows.append({"fracture_mode":name,"macro_F1":f1,"accuracy":acc})

frac_search=pd.DataFrame(frac_rows).sort_values(["macro_F1","accuracy"],ascending=False).reset_index(drop=True)
display(frac_search.head(20))

bf=frac_search.iloc[0]
if bf.fracture_mode=="current_raw":
    final_frac=current_frac
elif bf.fracture_mode=="off":
    final_frac=np.zeros(len(base),float)
else:
    th=float(str(bf.fracture_mode).split("_")[-1]); final_frac=(current_frac>=th).astype(float)

## 10. Final report

In [ ]:
base_f1,base_acc,_=evaluate(agg_b4["raw"],current_mls,current_frac)
fusion_f1,fusion_acc,_=evaluate(best_ich,current_mls,current_frac)
gate_f1,gate_acc,_=evaluate(gated_ich,current_mls,current_frac)
mls_f1,mls_acc,_=evaluate(gated_ich,best_mls,current_frac)
final_f1,final_acc,final_pred=evaluate(gated_ich,best_mls,final_frac)

summary=pd.DataFrame([
    {"stage":"B4 raw baseline","macro_F1":base_f1,"accuracy":base_acc},
    {"stage":"B4/B5 fusion","macro_F1":fusion_f1,"accuracy":fusion_acc},
    {"stage":"+ ICH presence gate","macro_F1":gate_f1,"accuracy":gate_acc},
    {"stage":"+ ordinal MLS","macro_F1":mls_f1,"accuracy":mls_acc},
    {"stage":"+ best fracture handling","macro_F1":final_f1,"accuracy":final_acc},
])
summary.to_csv(OUTPUT_ROOT/"00_DIRECT_ANSWERS.csv",index=False)
display(summary)

print("Best B4/B5 fusion:")
display(best.to_frame().T)
print("Best gate:")
display(bg.to_frame().T)
print("Best MLS:")
display(bm.to_frame().T)
print("Best fracture mode:",bf.fracture_mode)

## 11. Save frozen DEV config

In [ ]:
config={
    "ICH_fusion":{
        "profile_b4":str(best.profile_b4),
        "profile_b5":str(best.profile_b5),
        "alpha_b4":float(best.alpha_b4),
        "alpha_b5":float(best.alpha_b5),
    },
    "ICH_gate":{
        "aggregator":str(bg.aggregator),
        "threshold":float(bg.threshold),
        "volume_override_mL":float(bg.override_total_mL),
    },
    "MLS":{
        "agg3":str(bm.agg3),"threshold3":float(bm.th3),
        "agg5":str(bm.agg5),"threshold5":float(bm.th5),
    },
    "fracture_mode":str(bf.fracture_mode),
    "DEV_macro_F1":float(final_f1),
    "DEV_accuracy":float(final_acc),
    "locked_test_used_for_tuning":False,
}
with open(OUTPUT_ROOT/"fusion_config.json","w",encoding="utf-8") as f:
    json.dump(config,f,indent=2)
print(json.dumps(config,indent=2))